In [ ]:
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import random
import logging

# Configurar logs (evidencia para el informe)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
log = logging.getLogger("pipeline_fraude")

log.info("Iniciando pipeline de detección de fraude")

np.random.seed(42)
random.seed(42)

N_LEGIT  = 9000
N_FRAUD  = 500

# Datos adicionales para completar las columnas requeridas
nombres    = ['John','Maria','Carlos','Ana','Luis','Sofia','Pedro','Laura']
apellidos  = ['Smith','Garcia','Lopez','Martinez','Brown','Wilson','Taylor']
ciudades   = [('New York','NY','10001'), ('Los Angeles','CA','90001'),
              ('Chicago','IL','60601'), ('Houston','TX','77001'),
              ('Phoenix','AZ','85001'), ('Dallas','TX','75201')]

# --- Transacciones legítimas ---
for i in range(N_LEGIT):
    dt   = fecha_aleatoria(inicio, fin)
    lat  = round(random.uniform(25, 48), 4)
    lon  = round(random.uniform(-120, -70), 4)
    cat  = random.choice(categorias)
    amt  = round(random.uniform(5, 300), 2)
    ciudad, estado, zip_base = random.choice(ciudades)

    # Fecha de nacimiento formato DD-MM-AAAA
    anio = random.randint(1950, 2000)
    mes  = random.randint(1, 12)
    dia  = random.randint(1, 28)
    dob  = f"{dia:02d}-{mes:02d}-{anio}"

    filas.append({
        # Identidad
        'first'                : random.choice(nombres),
        'last'                 : random.choice(apellidos),
        'dob'                  : dob,
        'gender'               : random.choice(['M','F']),
        'job'                  : random.choice(jobs),
        # Ubicación
        'street'               : f"{random.randint(100,999)} Main St",
        'city'                 : ciudad,
        'state'                : estado,
        'zip'                  : int(zip_base) + random.randint(0, 99),
        'city_pop'             : random.randint(50_000, 3_000_000),
        'lat'                  : lat,
        'long'                 : lon,
        # Transacción
        'trans_num'            : f"txn_{i:06d}",
        'trans_date_trans_time': dt.strftime('%Y-%m-%d %H:%M:%S'),
        'unix_time'            : int(dt.timestamp()),
        'amt'                  : amt,
        'cc_num'               : random.randint(10**15, 10**16 - 1),
        # Comercio
        'merchant'             : random.choice(merch_legit),
        'category'             : cat,
        'merch_lat'            : round(lat + random.uniform(-0.5, 0.5), 4),
        'merch_long'           : round(lon + random.uniform(-0.5, 0.5), 4),
        # Target
        'is_fraud'             : 0
    })

# --- Transacciones fraudulentas ---
for i in range(N_FRAUD):
    dt   = fecha_aleatoria(inicio, fin)
    dt   = dt.replace(hour=random.choice([0,1,2,3,23]))
    lat  = round(random.uniform(25, 48), 4)
    lon  = round(random.uniform(-120, -70), 4)
    ciudad, estado, zip_base = random.choice(ciudades)

    anio = random.randint(1950, 2000)
    mes  = random.randint(1, 12)
    dia  = random.randint(1, 28)
    dob  = f"{dia:02d}-{mes:02d}-{anio}"

    filas.append({
        # Identidad
        'first'                : random.choice(nombres),
        'last'                 : random.choice(apellidos),
        'dob'                  : dob,
        'gender'               : random.choice(['M','F']),
        'job'                  : random.choice(jobs),
        # Ubicación
        'street'               : f"{random.randint(100,999)} Main St",
        'city'                 : ciudad,
        'state'                : estado,
        'zip'                  : int(zip_base) + random.randint(0, 99),
        'city_pop'             : random.randint(50_000, 3_000_000),
        'lat'                  : lat,
        'long'                 : lon,
        # Transacción
        'trans_num'            : f"fraud_{i:06d}",
        'trans_date_trans_time': dt.strftime('%Y-%m-%d %H:%M:%S'),
        'unix_time'            : int(dt.timestamp()),
        'amt'                  : round(random.uniform(800, 5000), 2),
        'cc_num'               : random.randint(10**15, 10**16 - 1),
        # Comercio
        'merchant'             : random.choice(merch_fraud),
        'category'             : random.choice(['shopping_net','misc_net','travel']),
        'merch_lat'            : round(random.uniform(25, 48), 4),
        'merch_long'           : round(random.uniform(-120, -70), 4),
        # Target
        'is_fraud'             : 1
    })

df_raw = pd.DataFrame(filas).sample(frac=1).reset_index(drop=True)

CARPETA = '/content/drive/MyDrive/Colab Notebooks/Gestion_Datos_IA_EV2'
df_raw.to_csv(f'{CARPETA}/datos_generados.csv', index=False)

log.info(f"Dataset guardado en Drive: {df_raw.shape}")
log.info(f"Dataset generado: {len(df_raw)} filas")
log.info(f"Distribución: {df_raw['is_fraud'].value_counts().to_dict()}")
log.info(f"Columnas: {list(df_raw.columns)}")

df_raw.head()

,trans_date_trans_time,unix_time,cc_num,merchant,category,amt,gender,city_pop,job,lat,...,merch_long,is_fraud,first,last,dob,street,city,state,zip,trans_num
0,2023-03-14 10:56:05,1678791365,9396320034735130,Walmart,shopping_net,140.14,F,1836746,Engineer,35.8037,...,-77.3797,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-06-27 18:31:54,1687890714,6649295292390007,Netflix,food_dining,152.23,F,1926676,Manager,40.7899,...,-78.8695,0,Pedro,Lopez,14-02-1960,208 Main St,Los Angeles,CA,90006.0,txn_002551
2,2023-03-16 07:02:21,1678950141,2438468161503935,Walmart,misc_net,271.33,M,1939988,Doctor,41.6915,...,-110.1858,0,Sofia,Garcia,27-01-1950,806 Main St,Houston,TX,77061.0,txn_007853
3,2023-08-26 21:27:40,1693085260,2231648692486389,Shell,travel,157.43,M,2745742,Nurse,46.3621,...,-93.7921,0,Maria,Wilson,25-05-1963,671 Main St,Chicago,IL,60678.0,txn_008373
4,2023-05-11 01:27:26,1683768446,6810576001665573,Netflix,food_dining,22.95,M,2970269,Accountant,36.6543,...,-95.9002,0,Ana,Martinez,22-10-1956,719 Main St,Los Angeles,CA,90007.0,txn_008996
